In [48]:
import os
from dotenv import load_dotenv

# Load key from .env file or set directly
load_dotenv()

# Verify key exists
if not os.environ.get("GEMINI_API_KEY"):
    import getpass
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter your GEMINI_API_KEY: ")

print("GEMINI_API_KEY configured.")

GEMINI_API_KEY configured.


# Cell 2: Imports, Pydantic Schema & Chain Setup

In [ ]:
from typing import List, Literal, Tuple
import pandas as pd
from IPython.display import display, HTML
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

FeatureState = Literal["Yes", "No", "Blank"]

class FeatureAuditResult(BaseModel):
    f1_individual_questions: FeatureState = Field(
        description="Yes: asked questions individually / Questions asked individually by name / included everyone / Maintained one to one queries. No: Generic group questions only. Blank: Neutral."
    )
    f1_evidence: str = Field(description="Direct transcript quote or observation.")

    f2_homework_given: FeatureState = Field(
        description="Yes: HW was given / assigned homework / gave homework. No: missed HW / class ended without homework. Blank: Not evaluated."
    )
    f2_evidence: str = Field(description="Direct transcript quote or observation.")

    f_rapport_building: FeatureState = Field(
        description="Yes: Good rapport building (informal check-ins, personal banter). No: Skipped rapport / Directly entered class without building rapport. Blank: Minimal/standard greeting."
    )
    f_rapport_evidence: str = Field(description="Direct transcript quote or observation.")

    f3_solid_intro: FeatureState = Field(
        description="Yes: Included a solid introduction / Good intro. No: Lacked good intro. Blank: Neutral."
    )
    f3_evidence: str = Field(description="Direct transcript quote or observation.")

    f4_setup_and_puzzles: FeatureState = Field(
        description="Yes: Addressed learners during setup and puzzles. No: Silent or left learners unaddressed during exercises. Blank: No setup/puzzle phase observed."
    )
    f4_evidence: str = Field(description="Direct transcript quote or observation.")

    f5_interactive_flow: FeatureState = Field(
        description="Yes: General interactive flow was present (continuous 2-way dialogue). No: Long tutor monologues / passive silence. Blank: Neutral."
    )
    f5_evidence: str = Field(description="Direct transcript quote or observation.")

    f6_conclusion_quality: FeatureState = Field(
        description="No: Lacked conclusion (abrupt end, no wrap-up). Yes: Clear wrap-up provided. Blank: Standard end."
    )
    f6_evidence: str = Field(description="Direct transcript quote or observation.")

    f7_joiner_intros: FeatureState = Field(
        description="Yes: joiner introductions (allowed time for new student to introduce themselves). No: missed joiner intro / lacked joiner introductions. Blank: No new joiners in session."
    )
    f7_evidence: str = Field(description="Direct transcript quote or observation.")

    students_called: List[str] = Field(
        description="Names of individual students called upon during the class."
    )

EXTRACTION_SYSTEM_PROMPT = """
You are an expert pedagogical auditor for online children's chess classes.
Analyze the provided transcript and extract the state for each of the 7 features strictly as 'Yes', 'No', or 'Blank', providing exact transcript evidence quotes:

1. Feature 1 (Individual Questions):
   - 'Yes': asked questions individually / Questions asked individually by name / included everyone / Maintained one to one queries.
   - 'No': Failed to query individually, addressed only the whole room, or left learners out.
   - 'Blank': Neutral / not evaluated.

2. Feature 2 (Homework):
   - 'Yes': HW was given / assigned homework / gave homework.
   - 'No': missed HW / should give HW (session ended without assigning homework tasks).
   - 'Blank': Not addressed.

3. Rapport Building:
   - 'Yes': Good rapport building (warm casual check-ins, non-chess icebreakers, friendly banter).
   - 'No': Skipped rapport / Directly entered class without building rapport (dived straight into revision/moves).
   - 'Blank': Standard neutral greeting.

4. Feature 3 (Introduction):
   - 'Yes': Included a solid introduction / Good intro (clear orientation of topics/goals).
   - 'No': Lacked good intro.
   - 'Blank': Neutral.

5. Feature 4 (Setup & Puzzles):
   - 'Yes': Addressed learners during setup and puzzles (actively coached learners while setting up or solving).
   - 'No': Passive, left students waiting in silence without coaching.
   - 'Blank': No setup or puzzle phase.

6. Feature 5 (Interactive Flow):
   - 'Yes': General interactive flow was present (active back-and-forth dialogue throughout).
   - 'No': Long monologue by tutor, one-sided interaction.
   - 'Blank': Neutral.

7. Feature 6 (Conclusion):
   - 'No': Lacked conclusion (abrupt drop-off without summarizing or closing).
   - 'Yes': Proper session wrap-up given.
   - 'Blank': Standard exit.

8. Feature 7 (Joiner Introductions):
   - 'Yes': joiner introductions (facilitated dedicated time for new students to introduce themselves).
   - 'No': missed joiner intro / lacked joiner introductions (new student was present but not invited to introduce themselves).
   - 'Blank': No new joiners attended.
"""

llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    google_api_key=os.environ["GEMINI_API_KEY"],
    temperature=0.0
)

structured_extractor = llm.with_structured_output(FeatureAuditResult)
prompt = ChatPromptTemplate.from_messages([
    ("system", EXTRACTION_SYSTEM_PROMPT),
    ("human", "Class Transcript:\n\n{transcript_text}")
])
extraction_chain = prompt | structured_extractor
print("Extraction chain initialized successfully.")

Extraction chain initialized successfully.


In [56]:
import os
import re
from typing import List, Literal, Tuple
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from IPython.display import display, HTML

FeatureState = Literal["Yes", "No", "Blank"]

# 1. Schema
class FeatureAuditResult(BaseModel):
    f1_individual_questions: FeatureState = Field(
        description="Yes: Asked questions individually by name, included at least 2 distinct students. No: Addressed only the group or only 1 dominant student. Blank: Neutral."
    )
    f1_evidence: str
    f2_homework_given: FeatureState = Field(
        description="Yes: Homework or practice puzzles explicitly assigned. No: Class ended without assigning practice tasks. Blank: Neutral."
    )
    f2_evidence: str
    f_rapport_building: FeatureState = Field(
        description="Yes: Warm non-chess personal banter/check-ins. No: Jumps straight into revision or board within 2 mins. Blank: Standard neutral greeting."
    )
    f_rapport_evidence: str
    f3_solid_intro: FeatureState = Field(
        description="Yes: Provided clear session goals and topic orientation. No: Lacked good intro. Blank: Neutral."
    )
    f3_evidence: str
    f4_setup_and_puzzles: FeatureState = Field(
        description="Yes: Actively coached during puzzles/setup. No: Silent or left students unguided. Blank: No puzzle phase."
    )
    f4_evidence: str
    f5_interactive_flow: FeatureState = Field(
        description="Yes: Steady back-and-forth dialogue. No: Monologues or dead air. Blank: Neutral."
    )
    f5_evidence: str
    f6_conclusion_quality: FeatureState = Field(
        description="No: Abrupt end without wrap-up. Yes: Clean summary/closing. Blank: Neutral."
    )
    f6_evidence: str
    f7_joiner_intros: FeatureState = Field(
        description="Yes: Dedicated time for new joiner to introduce themselves. No: New joiner present but ignored. Blank: No new joiners."
    )
    f7_evidence: str
    students_called: List[str]

# 2. Strict Extraction Prompt
EXTRACTION_SYSTEM_PROMPT = """You are an expert pedagogical auditor evaluating online chess transcripts.
Classify each feature strictly as 'Yes', 'No', or 'Blank' with direct quotes:
- f1_individual_questions: 'Yes' ONLY if tutor calls at least two different students by name to answer. If whole-group only, mark 'No'.
- f2_homework_given: 'Yes' ONLY if explicit home practice/puzzles are assigned. If the session ends without homework, mark 'No'.
- f_rapport_building: 'Yes' ONLY if there is authentic non-chess social check-in (school, weekend, hobbies). If tutor jumps straight into moves/revision, mark 'No'. Brief 'hi' is 'Blank'.
- f7_joiner_intros: 'Yes' if a new student introduced themselves. 'No' if a first-day student was present but skipped. 'Blank' if no new joiners.
"""

# 3. Model Setup (Groq)
llm = ChatGroq(
    model_name="openai/gpt-oss-20b",
    groq_api_key=os.environ.get("GROQ_API_KEY"),
    temperature=0.0
)

structured_extractor = llm.with_structured_output(FeatureAuditResult)
prompt = ChatPromptTemplate.from_messages([
    ("system", EXTRACTION_SYSTEM_PROMPT),
    ("human", "Class Transcript Dialogue:\n\n{transcript_text}")
])
extraction_chain = prompt | structured_extractor

# 4. Transcript Windowing (Prevents TPM errors on Groq)
def extract_compact_payload(raw_vtt: str) -> str:
    lines = [
        l.strip() for l in raw_vtt.splitlines() 
        if l.strip() and not l.startswith("WEBVTT") and not l.isdigit() and "-->" not in l
    ]
    full_text = "\n".join(lines)
    opening = full_text[:4000]
    closing = full_text[-3500:]
    mid_sample = [
        l for l in full_text[4000:-3500].splitlines()
        if any(w in l.lower() for w in ["?", "puzzle", "move", "can you", "what is"])
    ][:35]
    return f"{opening}\n\n[... mid-session queries & exercises ...]\n" + "\n".join(mid_sample) + f"\n\n[... closing segment ...]\n{closing}"

# 5. Anti-Bias Scoring Logic
def compute_score_and_rule(audit: FeatureAuditResult) -> Tuple[int, str, str]:
    f1 = audit.f1_individual_questions
    f2 = audit.f2_homework_given
    rap = audit.f_rapport_building
    f3 = audit.f3_solid_intro
    f4 = audit.f4_setup_and_puzzles
    f5 = audit.f5_interactive_flow
    f6 = audit.f6_conclusion_quality
    f7 = audit.f7_joiner_intros

    # VETO GATE: Check Low Failures First
    if rap == "No" and f7 == "No":
        return 1, "Low Value", "Skipped rapport for revision + missed joiner intro"
    if rap == "No":
        return 1, "Low Value", "Directly entered class without building rapport"
    if f3 == "No" and f6 == "No":
        return 1, "Low Value", "Lacked good intro/conclusion and joiner introductions"

    # HIGH TIER (Score 4) - Missing HW strictly blocks Score 4
    if f2 != "No":
        if f1 == "Yes" and f2 == "Yes":
            return 4, "High Value", "Questions asked individually by name + HW was given"
        if rap == "Yes" and f1 == "Yes":
            return 4, "High Value", "Good rapport building + asked questions individually"

    # MEDIUM TIER (Score 3)
    if f3 == "Yes" and f1 == "Yes" and f2 == "No":
        return 3, "Medium Value", "Good intro and included everyone, but missed HW"
    if f1 == "Yes" and f2 == "No":
        return 3, "Medium Value", "Maintained one-to-one queries, but should give HW"
    if f3 == "Yes" and f2 == "Yes":
        return 3, "Medium Value", "Included a solid introduction and assigned HW"
    if f4 == "Yes" and f2 == "Yes":
        return 3, "Medium Value", "Addressed learners during setup and puzzles; gave HW"
    if f5 == "Yes":
        return 3, "Medium Value", "General interactive flow was present"

    return 3, "Medium Value", "Standard baseline flow"

# 6. Display Function
def audit_and_display(vtt_file_path: str):
    with open(vtt_file_path, "r", encoding="utf-8", errors="ignore") as f:
        compact_text = extract_compact_payload(f.read())

    extracted = extraction_chain.invoke({"transcript_text": compact_text})
    score, tier, rule = compute_score_and_rule(extracted)

    score_color = "#27ae60" if score == 4 else ("#e67e22" if score == 3 else "#c0392b")
    
    table_rows = [
        ("1", "Questions Asked Individually", extracted.f1_individual_questions, extracted.f1_evidence),
        ("2", "Homework Assignment", extracted.f2_homework_given, extracted.f2_evidence),
        ("—", "Rapport Building", extracted.f_rapport_building, extracted.f_rapport_evidence),
        ("3", "Introduction Quality", extracted.f3_solid_intro, extracted.f3_evidence),
        ("4", "Setup & Puzzles Engagement", extracted.f4_setup_and_puzzles, extracted.f4_evidence),
        ("5", "General Interactive Flow", extracted.f5_interactive_flow, extracted.f5_evidence),
        ("6", "Session Conclusion", extracted.f6_conclusion_quality, extracted.f6_evidence),
        ("7", "Joiner Introductions", extracted.f7_joiner_intros, extracted.f7_evidence),
    ]

    rows_html = ""
    for num, name, state, ev in table_rows:
        if state == "Yes":
            badge = '<span style="background: #e2efda; color: #276a3c; font-weight: bold; padding: 2px 8px; border-radius: 4px;">Yes</span>'
        elif state == "No":
            badge = '<span style="background: #fce4d6; color: #c65911; font-weight: bold; padding: 2px 8px; border-radius: 4px;">No</span>'
        else:
            badge = '<span style="color: #7f8c8d; padding: 2px 8px;">—</span>'
        
        rows_html += f"""
        <tr style="border-bottom: 1px solid #e5e7eb;">
            <td style="padding: 8px; text-align: center; font-weight: bold;">{num}</td>
            <td style="padding: 8px; font-weight: 500;">{name}</td>
            <td style="padding: 8px; text-align: center;">{badge}</td>
            <td style="padding: 8px; font-size: 13px; color: #374151;">{ev}</td>
        </tr>
        """

    display_html = f"""
    <div style="font-family: Arial, sans-serif; margin-bottom: 20px;">
        <div style="border-left: 5px solid {score_color}; padding: 10px 14px; background: #ffffff; box-shadow: 0 1px 3px rgba(0,0,0,0.1); border-radius: 4px; margin-bottom: 10px;">
            <h4 style="margin: 0; color: #111827;">File: {os.path.basename(vtt_file_path)}</h4>
            <p style="margin: 4px 0 0 0; font-size: 14px;">
                <b>Score:</b> <span style="color: {score_color}; font-weight: bold; font-size: 16px;">{score} / 4</span> ({tier}) &nbsp;|&nbsp; 
                <b>Rule:</b> <i>"{rule}"</i>
            </p>
            <p style="margin: 3px 0 0 0; font-size: 13px; color: #4b5563;">
                <b>Students Identified:</b> {', '.join(extracted.students_called) if extracted.students_called else 'None'}
            </p>
        </div>
        <table style="width: 100%; border-collapse: collapse; border: 1px solid #e5e7eb; font-size: 13.5px;">
            <thead>
                <tr style="background: #1f4e79; color: white; text-align: left;">
                    <th style="padding: 8px; width: 40px; text-align: center;">#</th>
                    <th style="padding: 8px; width: 220px;">Feature</th>
                    <th style="padding: 8px; width: 80px; text-align: center;">State</th>
                    <th style="padding: 8px;">Evidence Quote</th>
                </tr>
            </thead>
            <tbody>
                {rows_html}
            </tbody>
        </table>
    </div>
    """
    display(HTML(display_html))
    return extracted, score, rule

# Cell 3: Scoring Combinations & VTT Parser

In [57]:
def compute_score_and_rule(audit: FeatureAuditResult) -> Tuple[int, str, str]:
    f1 = audit.f1_individual_questions
    f2 = audit.f2_homework_given
    rap = audit.f_rapport_building
    f3 = audit.f3_solid_intro
    f4 = audit.f4_setup_and_puzzles
    f5 = audit.f5_interactive_flow
    f6 = audit.f6_conclusion_quality
    f7 = audit.f7_joiner_intros

    # High Value Feature Combination = 4
    if (f1 == "Yes" and f2 == "Yes") or (rap == "Yes" and f1 == "Yes"):
        rule = "Questions asked individually by name + HW was given" if (f1 == "Yes" and f2 == "Yes") else "Good rapport building + asked questions individually"
        return 4, "High Value", rule

    # Low Value Features Combination = 1
    if ((f3 == "No" and f6 == "No" and f7 == "Yes") or (rap == "No" and f7 == "No") or (rap == "No")):
        if f3 == "No" and f6 == "No" and f7 == "Yes":
            rule = "Lacked good intro + lacked conclusion + joiner introductions"
        elif rap == "No" and f7 == "No":
            rule = "Skipped rapport + missed joiner intro"
        else:
            rule = "Directly entered class without building rapport"
        return 1, "Low Value", rule

    # Medium Value Feature Combination = 3
    if f3 == "Yes" and f2 == "Yes":
        rule = "Included a solid introduction + assigned homework"
    elif f3 == "Yes" and f1 == "Yes" and f2 == "No":
        rule = "Good intro + included everyone + missed HW"
    elif f4 == "Yes" and f2 == "Yes":
        rule = "Addressed learners during setup and puzzles + gave homework"
    elif f1 == "Yes" and f2 == "No":
        rule = "Maintained one to one queries + missed Homeworks"
    else:
        rule = "General interactive flow was present"

    return 3, "Medium Value", rule

def clean_vtt(raw_text: str) -> str:
    lines = []
    for line in raw_text.splitlines():
        t = line.strip()
        if not t or t.startswith("WEBVTT") or t.isdigit() or "-->" in t:
            continue
        lines.append(t)
    return "\n".join(lines)

# cell 4

In [58]:
def audit_and_display(vtt_file_path: str):
    """Audits a class VTT file and displays a styled HTML table directly in the notebook without needing jinja2."""
    with open(vtt_file_path, "r", encoding="utf-8", errors="ignore") as f:
        clean_text = clean_vtt(f.read())
        
    extracted: FeatureAuditResult = extraction_chain.invoke({"transcript_text": clean_text})
    score, tier, rule = compute_score_and_rule(extracted)
    
    table_data = [
        ("1", "Questions Asked Individually", extracted.f1_individual_questions, extracted.f1_evidence),
        ("2", "Homework Assignment", extracted.f2_homework_given, extracted.f2_evidence),
        ("—", "Rapport Building", extracted.f_rapport_building, extracted.f_rapport_evidence),
        ("3", "Introduction Quality", extracted.f3_solid_intro, extracted.f3_evidence),
        ("4", "Setup & Puzzles Engagement", extracted.f4_setup_and_puzzles, extracted.f4_evidence),
        ("5", "General Interactive Flow", extracted.f5_interactive_flow, extracted.f5_evidence),
        ("6", "Session Conclusion", extracted.f6_conclusion_quality, extracted.f6_evidence),
        ("7", "Joiner Introductions", extracted.f7_joiner_intros, extracted.f7_evidence),
    ]
    
    score_color = "#27ae60" if score == 4 else ("#e67e22" if score == 3 else "#c0392b")
    
    # Build Table Rows
    rows_html = ""
    for idx, (num, name, state, ev) in enumerate(table_data):
        bg_row = "#ffffff" if idx % 2 == 0 else "#f9fafb"
        
        # State Badge Styling
        if state == "Yes":
            badge = '<span style="background: #e2efda; color: #276a3c; font-weight: bold; padding: 3px 8px; border-radius: 4px;">Yes</span>'
        elif state == "No":
            badge = '<span style="background: #fce4d6; color: #c65911; font-weight: bold; padding: 3px 8px; border-radius: 4px;">No</span>'
        else:
            badge = '<span style="color: #7f8c8d; font-weight: bold; padding: 3px 8px;">—</span>'
            
        rows_html += f"""
        <tr style="background: {bg_row}; border-bottom: 1px solid #e5e7eb;">
            <td style="padding: 10px; text-align: center; color: #555; font-weight: bold;">{num}</td>
            <td style="padding: 10px; font-weight: bold; color: #1f2937;">{name}</td>
            <td style="padding: 10px; text-align: center;">{badge}</td>
            <td style="padding: 10px; color: #374151; font-size: 13px;">{ev}</td>
        </tr>
        """
        
    full_html = f"""
    <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; margin-bottom: 25px;">
        <div style="border-left: 5px solid {score_color}; padding: 12px 18px; margin-bottom: 12px; background: #ffffff; border-radius: 4px; box-shadow: 0 1px 3px rgba(0,0,0,0.08);">
            <h3 style="margin: 0; color: #1f2937; font-size: 16px;">File: <span style="font-weight: normal; color: #4b5563;">{os.path.basename(vtt_file_path)}</span></h3>
            <p style="margin: 6px 0 0 0; font-size: 15px;">
                <b>Final Score:</b> <span style="color: {score_color}; font-size: 18px; font-weight: bold;">{score} / 4</span> ({tier}) &nbsp;|&nbsp; 
                <b>Matched Rule:</b> <i>"{rule}"</i>
            </p>
            <p style="margin: 5px 0 0 0; font-size: 13px; color: #4b5563;">
                <b>Students Addressed Individually:</b> {', '.join(extracted.students_called) if extracted.students_called else 'None Identified'}
            </p>
        </div>
        
        <table style="width: 100%; border-collapse: collapse; border: 1px solid #e5e7eb; border-radius: 4px; overflow: hidden; font-size: 13.5px;">
            <thead>
                <tr style="background: #1f4e79; color: #ffffff; text-align: left;">
                    <th style="padding: 10px; width: 45px; text-align: center;">#</th>
                    <th style="padding: 10px; width: 230px;">Feature Metric</th>
                    <th style="padding: 10px; width: 90px; text-align: center;">State</th>
                    <th style="padding: 10px;">Observation / Quoted Evidence</th>
                </tr>
            </thead>
            <tbody>
                {rows_html}
            </tbody>
        </table>
    </div>
    """
    
    display(HTML(full_html))
    return extracted, score, rule

# Cell 5

In [59]:
def audit_and_display(vtt_file_path: str):
    """Audits a class VTT file and displays a styled HTML table directly in the notebook."""
    with open(vtt_file_path, "r", encoding="utf-8", errors="ignore") as f:
        clean_text = clean_vtt(f.read())
        
    extracted: FeatureAuditResult = extraction_chain.invoke({"transcript_text": clean_text})
    score, tier, rule = compute_score_and_rule(extracted)
    
    # Build DataFrame
    table_data = [
        {"#": "1", "Feature Metric": "Questions Asked Individually", "State": extracted.f1_individual_questions, "Observation / Evidence": extracted.f1_evidence},
        {"#": "2", "Feature Metric": "Homework Assignment", "State": extracted.f2_homework_given, "Observation / Evidence": extracted.f2_evidence},
        {"#": "—", "Feature Metric": "Rapport Building", "State": extracted.f_rapport_building, "Observation / Evidence": extracted.f_rapport_evidence},
        {"#": "3", "Feature Metric": "Introduction Quality", "State": extracted.f3_solid_intro, "Observation / Evidence": extracted.f3_evidence},
        {"#": "4", "Feature Metric": "Setup & Puzzles Engagement", "State": extracted.f4_setup_and_puzzles, "Observation / Evidence": extracted.f4_evidence},
        {"#": "5", "Feature Metric": "General Interactive Flow", "State": extracted.f5_interactive_flow, "Observation / Evidence": extracted.f5_evidence},
        {"#": "6", "Feature Metric": "Session Conclusion", "State": extracted.f6_conclusion_quality, "Observation / Evidence": extracted.f6_evidence},
        {"#": "7", "Feature Metric": "Joiner Introductions", "State": extracted.f7_joiner_intros, "Observation / Evidence": extracted.f7_evidence},
    ]
    df = pd.DataFrame(table_data)
    
    # Color badge mapping
    def highlight_state(val):
        if val == "Yes":
            return "background-color: #e2efda; color: #276a3c; font-weight: bold; text-align: center;"
        elif val == "No":
            return "background-color: #fce4d6; color: #c65911; font-weight: bold; text-align: center;"
        return "color: #7f8c8d; text-align: center;"
    
    styled_df = df.style.map(highlight_state, subset=["State"]).set_properties(**{
        'text-align': 'left',
        'white-space': 'normal'
    })
    
    score_color = "#27ae60" if score == 4 else ("#e67e22" if score == 3 else "#c0392b")
    
    header_html = f"""
    <div style="border-left: 5px solid {score_color}; padding: 10px 15px; margin-bottom: 12px; background: #fdfdfd; border-radius: 4px; box-shadow: 0 1px 3px rgba(0,0,0,0.08);">
        <h3 style="margin: 0; color: #2c3e50;">File: <span style="font-weight: normal;">{os.path.basename(vtt_file_path)}</span></h3>
        <p style="margin: 5px 0 0 0; font-size: 15px;">
            <b>Final Score:</b> <span style="color: {score_color}; font-size: 18px; font-weight: bold;">{score} / 4</span> ({tier}) &nbsp;|&nbsp; 
            <b>Matched Rule:</b> <i>"{rule}"</i>
        </p>
        <p style="margin: 4px 0 0 0; font-size: 13px; color: #555;">
            <b>Students Addressed:</b> {', '.join(extracted.students_called) if extracted.students_called else 'None Identified'}
        </p>
    </div>
    """
    
    display(HTML(header_html))
    display(styled_df)
    return extracted, score, rule

In [ ]:
# Execute directly on your local sample transcript
sample_vtt = "B472 intermediate -transcript-1.1.vtt"

if os.path.exists(sample_vtt):
    extracted, score, rule = audit_and_display(sample_vtt)
else:
    print(f"File '{sample_vtt}' not found in current folder.")

BadRequestError: Error code: 400 - {'error': {'message': 'Tool choice is required, but model did not call a tool', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': ''}}

In [55]:
import os

# Set the key directly in the current session
os.environ["GEMINI_API_KEY"] = "AQ.Ab8RN6JNZUwnSUegxmgZKrQTxPdKMJxIhFa2aG4sYagub8ruFQ"

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    google_api_key=os.environ["GEMINI_API_KEY"],
    temperature=0.0
)

res = llm.invoke("Reply with 'API key verified'")
print("Result:", res.content)

GoogleModelNotFoundError: Error calling model 'gemini-2.0-flash' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.0-flash is no longer available. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://ai.google.dev/gemini-api/docs/get-started).', 'status': 'NOT_FOUND'}}